- 有数据时，先明确每行每列以及单位
- 检查缺失值
  - 异常值（离谱的，需判断真实极端还是录入错误）
  - 重复值（记录多次）
  - 数据类型错误（数字类型变成字符串有问题）
- 把不同单位，不同方向，不同尺度数据转换为可比较形式
- 通过统计量、图表、相关性分析，为模型选择提供依据
- 评委不仅看结果，还看数据来源、处理依据和是否可复现
- 目标：让数据被模型正确使用，将附件整理为可直接使用的数值矩阵

- X=[x_ij]_n*m
- n个评价对象，m个指标，x_ij表示第i个对象在第j个指标上的观测值

- 对象——指标——数据矩阵——模型输入——结果输出

- 注意：
  - 缺失值一定要处理，指标方向要统一（越大/小越好）
  - 量纲要消除

- 每一行称样本，每一列称指标，值为观测结果

- 常见数据格式：
  - Excel：`.xlsx` → `pd.read_excel(路径)`
  - CSV：`.csv` → `pd.read_csv(路径)`
- 图片文本先提取信息，一般不需要导入

- 读取数据检查顺序：
  - 文件是否存在
  - 编码是否正确（是否乱码，重下）
  - 列名是否完整
  - 数据类型是否合理
  - 是否可保存

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

#要先把数据放vscode里
DATA_PATH=Path('d:\\git项目\\导入数据')
OUTPUT_DIR=Path('d:\\git项目\\输出数据')

print(Path.cwd())#输出当前工作目录
print(DATA_PATH.exists())#判断导入路径是否存在
print(OUTPUT_DIR.exists())#判断输出路径是否存在


In [ ]:
#把附件变成DataFrame
df_raw=pd.read_csv(DATA_PATH/'初始资金为10000的资金变化.csv')
df_raw

In [ ]:
print(df_raw.shape)#输出数据的行列数
df_raw.info()#输出数据的基本信息

- 数据清洗学习
  - 1、把错误不完整不一致和异常记录进行检查和处理
    - 查看字段——检查确实——检查重复——检查类型——识别异常——记录处理
  - 2、清洗原则：说清楚每一步原因，并能在这里复现（用代码）
    - 原始附件——处理代码——输出表格
  - 3、论文中说明：处理对象、处理方法、处理依据，以及处理前后变化

- 缺失值常见来源：
  - 1、数据采集时没记录
  - 2、问卷中受访者未填写
  - 3、设备故障中断
  - 4、多个数据表合并字段不匹配
  - 5、原始数据中使用特殊符号表示缺测
- 影响均值、方差、相关系数、回归模型、聚类模型和综合评价结果
- 缺失值出现在某一类样本中时，随意删除可能造成样本偏差，处理缺失值关键在于保留数据含义并减少偏差

- 缺失率=缺失值个数/样本总数*100%
- 0、某一列缺失率高时可能直接删除列，不删除行
- 1、缺失很少且不影响样本代表性时删除行，但一般连续时间序列不可随便删去，可能会造成时间跨度不一致
- 2、缺失值少时可用均值、中位数、众数、分组中位数填补
  - （差异明显时按区域分组，计算中位数，填补缺失再检查）尽量尊重数据分组特征
- 3、时间序列或有顺序的数据可用插值（线性插值等）
- 4、缺失本身有含义不能机械填补要单独说明

In [ ]:
# -*- coding: utf-8 -*-
# 依赖：pip install pandas numpy openpyxl
import sys; print(sys.executable)   # 先确认 VSCode 正在用哪个解释器
%pip install openpyxl
from pathlib import Path
import numpy as np
import pandas as pd

IN_DIR = Path(r"d:\git项目\导入数据")
OUT_DIR = Path(r"d:\git项目\输出数据")
NAME = "mcm_missing_raw"              # 文件名（不含扩展名）
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_any(folder, stem, order=(".csv", ".xlsx", ".xls")):
    for ext in order:                 # 优先 csv，找不到再读 xlsx
        p = folder / f"{stem}{ext}"
        if p.exists():
            print("读取：", p)
            return pd.read_excel(p) if ext in (".xlsx", ".xls") else pd.read_csv(p)
    raise FileNotFoundError(f"{folder} 下找不到 {stem}.csv / .xlsx")

df = load_any(IN_DIR, NAME)

# 特殊符号转 NaN，数值列转数值
df = df.replace(["", "NA", "缺测", "未填", "9999", 9999], np.nan)
cols = ["pm25", "temperature", "traffic_index", "survey_score", "household_income"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")
rate = df.isna().mean() * 100         # 缺失率 = 缺失值个数 / 样本总数 ×100%
print("\n原始缺失率(%):\n" + rate.round(2).to_string())
print("\n分组缺失率(%):\n" + df.groupby("region")["pm25"]
      .apply(lambda s: s.isna().mean() * 100).round(2).to_string())

# 情况 0：列缺失率高 -> 删列，不删行
df = df.drop(columns=rate[rate >= 50].index)
print(f"\n[0] 删列后：{df.shape[0]} 行 {df.shape[1]} 列")

# 情况 1：横截面且缺失极少才删行；时间序列一律不删
is_time_series = True                 # 本例为逐日面板，横截面改 False
if is_time_series:
    print("[1] 时间序列：不删行，避免时间跨度不一致")
else:
    bad = df[["traffic_index", "fuel_type"]].isna().any(axis=1)
    df = df.dropna(subset=["traffic_index", "fuel_type"]) if bad.mean() < 0.05 else df

# 情况 2：均值 / 中位数 / 众数 / 分组中位数
print("[2] 均值 %.2f / 中位数 %.2f / 众数 %s" % (
    df["traffic_index"].mean(), df["traffic_index"].median(), df["fuel_type"].mode()[0]))
df["traffic_index"] = df["traffic_index"].fillna(df["traffic_index"].median())
df["fuel_type"] = df["fuel_type"].fillna(df["fuel_type"].mode()[0])
group_med = df.groupby("region")["pm25"].median()
g = df.groupby("region")["pm25"].transform("median")   # 尊重区域差异
df["pm25"] = df["pm25"].fillna(g).fillna(df["pm25"].median())
print("    分组中位数：", group_med.round(2).to_dict(),
      "| 填补后剩余缺失", df["pm25"].isna().sum())

# 情况 3：时间序列按分组 + 时间排序后线性插值
df = df.sort_values(["region", "date"])
df["temperature"] = df.groupby("region")["temperature"].transform(
    lambda s: s.interpolate(method="linear", limit_direction="both"))
print(f"[3] 插值后行数不变：{len(df)}")

# 情况 4：缺失本身有含义 -> 保留 NaN + 指示变量
df["survey_score_missing"] = df["survey_score"].isna().astype(int)
print(f"[4] 保留问卷拒答 {df['survey_score'].isna().sum()} 个，不做填补")

# 复核 + 输出两种格式
print("\n处理后缺失率(%):\n" + (df.isna().mean() * 100).round(2).to_string())
assert df[["pm25", "temperature", "traffic_index", "fuel_type"]].isna().sum().sum() == 0
df.to_csv(OUT_DIR / "mcm_missing_cleaned.csv", index=False, encoding="utf-8-sig")
df.to_excel(OUT_DIR / "mcm_missing_cleaned.xlsx", index=False, sheet_name="清洗结果")
print("\n已输出:", OUT_DIR / "mcm_missing_cleaned.csv", "和 .xlsx")
       

- df.duplicated()检查有无重复录入数据
- df.drop_duplicates()删除重复行

- 类型错误时常用
  - astype()
  - to_numeric()转为数值型
  - to_datatime()转为时间序列
- 建模前检查常用
  - df.info()信息
  - df.describe()描述性统计
  - df.isna().sum()计算缺失值总数
  - df.duplicated().sum()重复行总数

- 异常值：数据错误或真实存在极端情况
  - 1、录入错误：多输数字、单位填错、符号写反
  - 2、真实极端样本：超大城市、极端天气、特殊年份
  - 3、模型敏感点：数值本身没错，但是显著影响均值、回归线或聚类结果，一般用到的少
- 异常值不是一发现就删除，先判断为什么异常，是录入还是真实值异常（核心）
- 录入异常：通勤1200分钟，GDP多个0，通常需要修正和删除
- 真实极端样本：虽然极端，但可能是真实现象，需要解释，例如上海重庆交通复杂，通勤时间长正常

- 用3σ原则判断数据是否异常，所有数据应在μ±3σ之间
  - 1、计算μ均值σ标准差，注意标准差根号里面分母为n-1
  - 2、得到区域
  - 3、判断数据点是否位于±3σ区域

- IQR方法
- 箱线图把Q1,中位数，Q3上下界和异常点全部展示出来
- IQR=Q_3-Q_1
- Q_1为第一四分位数，Q_3为第三四分位数
- IQR表示中间50%数据的跨度
- 认为所有数据应该属于[Q_1-1.5IQR,Q_3+1.5IQR]
- 在区间之外视为异常值
- 由于IQR依赖四分位数，而不是均值标准差，因此不容易被极端值本身拉偏
- 数据比较少常用IQR

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Windows / macOS / Linux 中文字体回退，避免中文标题变方块
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "SimSun", "Arial Unicode MS", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False  # 正常显示负号

# 1. 造数据
rng = np.random.default_rng(2026)  # 固定随机种子，结果可复现
data = [
    np.concatenate([rng.normal(50, 8, 60), [88, 92, 15]]),   # A 组：较集中
    np.concatenate([rng.normal(62, 12, 60), [110, 20]]),     # B 组：中等离散
    np.concatenate([rng.normal(45, 5, 60), [70, 25]]),       # C 组：很集中
    np.concatenate([rng.normal(70, 15, 60), [30, 130, 135]]),  # D 组：很分散
]
labels = ["A 组", "B 组", "C 组", "D 组"]

#  2. 打印统计量（1.5×IQR 规则）
print("组别   Q1     中位数   Q3     IQR    异常值个数")
for name, values in zip(labels, data):
    q1, median, q3 = np.percentile(values, [25, 50, 75])
    iqr = q3 - q1
    outliers = int(np.count_nonzero((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)))
    print(f"{name:<5} {q1:6.2f} {median:8.2f} {q3:6.2f} {iqr:7.2f} {outliers:8d}")

#  3. 绘图
fig, ax = plt.subplots(figsize=(8.5, 5.2), dpi=120)

# 不传 labels/tick_labels 参数（不同 matplotlib 版本参数名有差异），统一用 set_xticks
bp = ax.boxplot(
    data,
    positions=range(1, len(data) + 1),
    widths=0.5,
    patch_artist=True,      # 允许给箱体填色
    showmeans=True,         # 显示均值
    showfliers=True,        # 显示异常值
    meanprops={"marker": "D", "markerfacecolor": "#2c3e50",
               "markeredgecolor": "#2c3e50", "markersize": 6},
    medianprops={"color": "#c0392b", "linewidth": 2},
    flierprops={"marker": "o", "markerfacecolor": "#e67e22",
                "markeredgecolor": "#e67e22", "markersize": 5, "alpha": 0.75},
)

colors = ["#8ecae6", "#a7d7a5", "#ffd28a", "#f4a6a6"]
for box, color in zip(bp["boxes"], colors):
    box.set(facecolor=color, edgecolor="#333333", linewidth=1.2, alpha=0.9)
for key in ("whiskers", "caps"):
    for line in bp[key]:
        line.set(color="#555555", linewidth=1.2)

ax.set_xticks(range(1, len(labels) + 1), labels)
ax.set_title("四组数据分布箱线图", fontsize=14, pad=12)
ax.set_xlabel("分组", fontsize=11)
ax.set_ylabel("数值", fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

legend_items = [
    Patch(facecolor="#a7d7a5", edgecolor="#333333", label="箱体：Q1 ~ Q3（IQR）"),
    Line2D([0], [0], color="#c0392b", lw=2, label="中位数"),
    Line2D([0], [0], linestyle="None", marker="D", markerfacecolor="#2c3e50",
           markeredgecolor="#2c3e50", markersize=6, label="均值"),
    Line2D([0], [0], linestyle="None", marker="o", markerfacecolor="#e67e22",
           markeredgecolor="#e67e22", markersize=5, label="异常值（>1.5×IQR）"),
]
ax.legend(handles=legend_items, loc="upper left", fontsize=9, framealpha=0.9)

fig.tight_layout()

#4. 保存并显示
out = Path(__file__).resolve().with_name("boxplot_demo.png")
#out = Path('d:\\git项目\\输出数据').resolve().with_name("boxplot_demo.png")
#out = Path('d:\\git项目\\输出数据\\text').resolve().with_name("boxplot_demo.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"图片已保存到：{out}")

- 先看中位数，再看箱体宽度，最后看是否存在离群点
- 箱体越宽说明波动越大；异常点需要结合业务背景判断
- 用途：发现异常值，比较不同组数据分布差异

- 异常值处理方式：
  - 1、删除：明确录入错误，删除后不影响样本代表性
  - 2、替换：有可靠修正值，或用分组统计量填补
  - 3、缩尾：把超过上下界值压缩到边界，降低极端值影响
  - 例xi*=min(xi,Q3+1.5IQR)表示压缩超过的异常值为上界
  - 4、保留并说明：异常值为真实样本，可保留并在论文中说明